# Fine-tuning BERT on GoEmotions (multi-label text emotion classification)

This notebook is the project's **transfer-learning component**. It fine-tunes a pretrained BERT model on the GoEmotions dataset and saves a model that can later be wrapped by the text module of `ie_models`. It is **not** connected to the interview interface yet.

**Steps**

1. Load GoEmotions and inspect its examples and labels.
2. Define the task: multi-label text emotion classification.
3. Create training, validation and test splits.
4. Load a pretrained BERT tokenizer and model for sequence classification.
5. Fine-tune, evaluate with multi-label metrics, and save the model and tokenizer.

**How to run**

- On Google Colab (recommended): upload this notebook, choose *Runtime > Change runtime type > T4 GPU*, then *Runtime > Run all*. The first code cell installs what Colab is missing, the second stops with a clear message if no GPU is attached, and the last cells download the model and results to your computer.
- Locally: install `requirements.txt` (see `README.md`), start Jupyter from `model-training/text/`, and run all cells.
- `SMALL_RUN = True` (the default) trains on a small subset to check the pipeline end to end (a few minutes on a GPU). Set it to `False` for the full dataset (roughly 15 to 25 minutes per epoch on a Colab T4).

**Outputs** (relative to `model-training/text/` locally, or `/content/` on Colab)

| Path | Contents | In git? |
|---|---|---|
| `outputs/<run>/model/` | fine-tuned weights, config with label names, tokenizer, `inference_config.json` | no (large) |
| `outputs/<run>/checkpoints/` | Trainer checkpoints | no |
| `results/<run>_metrics.json` | config, data sizes, validation and test metrics, library versions | yes |
| `results/<run>_per_label.csv` | per-emotion precision, recall, F1 on the test split | yes |
| `downloads/<run>.zip` | `model/` + `results/` in one file, downloaded automatically on Colab | no |

`<run>` is `bert-goemotions-small` or `bert-goemotions-full`.

## 0. Environment and configuration

In [ ]:
# On Colab, install the dependencies. Locally, install requirements.txt instead.
import sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    %pip install -q "transformers>=4.46" "datasets>=2.20" "accelerate>=0.34" scikit-learn

In [ ]:
import json
import math
import platform
import time
from datetime import datetime, timezone
from pathlib import Path

import datasets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch
import transformers
from datasets import load_dataset
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    hamming_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)

# ---- Run configuration -------------------------------------------------------
SMALL_RUN = True  # True: quick pipeline check on a subset. False: full dataset.

CONFIG = {
    'model_name': 'bert-base-uncased',
    'dataset_name': 'google-research-datasets/go_emotions',
    'dataset_config': 'simplified',  # 27 emotions + neutral, official train/validation/test splits
    'max_length': 128,
    'learning_rate': 2e-5,
    'weight_decay': 0.01,
    'warmup_fraction': 0.1,
    'train_batch_size': 32,
    'eval_batch_size': 64,
    'num_epochs': 4,
    'early_stopping_patience': 2,
    'default_threshold': 0.5,
    'train_subset': None,  # None = use the whole split
    'eval_subset': None,
    'seed': 42,
}
if SMALL_RUN:
    # Few update steps, so use a higher learning rate and more epochs than the full run.
    CONFIG.update({'train_subset': 2000, 'eval_subset': 500, 'num_epochs': 5, 'learning_rate': 5e-5, 'max_length': 64})

RUN_NAME = f"bert-goemotions-{'small' if SMALL_RUN else 'full'}"

# Keep outputs next to this notebook, also when Jupyter is started from the repo root.
BASE_DIR = Path.cwd()
if (BASE_DIR / 'model-training' / 'text').is_dir():
    BASE_DIR = BASE_DIR / 'model-training' / 'text'
OUTPUT_DIR = BASE_DIR / 'outputs' / RUN_NAME
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'
MODEL_DIR = OUTPUT_DIR / 'model'
RESULTS_DIR = BASE_DIR / 'results'
for d in (CHECKPOINT_DIR, MODEL_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

set_seed(CONFIG['seed'])
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

ENV_INFO = {
    'python': platform.python_version(),
    'torch': torch.__version__,
    'transformers': transformers.__version__,
    'datasets': datasets.__version__,
    'scikit-learn': sklearn.__version__,
    'device': torch.cuda.get_device_name(0) if DEVICE == 'cuda' else platform.processor() or 'cpu',
}
print('Run:', RUN_NAME)
print('Outputs:', OUTPUT_DIR.resolve())
print(json.dumps(ENV_INFO, indent=2))

In [ ]:
# Check that Colab gave us a GPU before training starts.
if DEVICE == 'cuda':
    print('GPU ready:', torch.cuda.get_device_name(0))
elif IN_COLAB:
    raise RuntimeError(
        'No GPU found. In Colab choose Runtime > Change runtime type > T4 GPU, '
        'then Runtime > Run all again.'
    )
else:
    print('No GPU found: training will run on the CPU (only practical for the small run).')

## 1. Load GoEmotions and inspect examples and labels

GoEmotions (Demszky et al., 2020) has about 58k English Reddit comments labelled by human raters with 27 emotion categories plus *neutral*. The `simplified` configuration keeps the labels that raters agreed on and comes with an official 80/10/10 train/validation/test split.

In [ ]:
raw = load_dataset(CONFIG['dataset_name'], CONFIG['dataset_config'])
print(raw)

LABEL_NAMES = raw['train'].features['labels'].feature.names
NUM_LABELS = len(LABEL_NAMES)
print(f'\n{NUM_LABELS} labels:', ', '.join(LABEL_NAMES))

In [ ]:
# A few raw examples: each comment has a list of label ids.
for ex in raw['train'].select(range(10)):
    names = [LABEL_NAMES[i] for i in ex['labels']]
    print(f"{str(names):<40} | {ex['text']}")

In [ ]:
train_df = raw['train'].to_pandas()

# How often each emotion appears in the training split (the classes are very imbalanced).
label_counts = pd.Series(
    [LABEL_NAMES[i] for labels in train_df['labels'] for i in labels]
).value_counts()
label_table = label_counts.to_frame('examples')
label_table['% of comments'] = (100 * label_table['examples'] / len(train_df)).round(2)
display(label_table)

ax = label_counts.sort_values().plot.barh(figsize=(7, 8), title='GoEmotions train split: comments per label')
ax.set_xlabel('comments')
plt.tight_layout()
plt.show()

In [ ]:
# Labels per comment: this is why the task is multi-label.
labels_per_comment = train_df['labels'].apply(len)
display(labels_per_comment.value_counts().sort_index().rename_axis('labels per comment').to_frame('comments'))
print(f'{(labels_per_comment > 1).mean():.1%} of training comments have more than one label.')

multi = train_df[labels_per_comment > 1].head(5)
for _, row in multi.iterrows():
    print([LABEL_NAMES[i] for i in row['labels']], '|', row['text'])

# Comment length in words, to choose max_length for the tokenizer.
print('\nWords per comment:')
print(train_df['text'].str.split().str.len().describe(percentiles=[0.5, 0.9, 0.99]).round(1))

## 2. Task definition: multi-label text emotion classification

- **Input:** one piece of text (a comment; later, a transcribed interview answer).
- **Output:** an independent probability for each of the 28 labels. A comment may have **more than one** emotion, so this is *multi-label*, not multi-class: the probabilities do not sum to 1.
- **Targets:** each example's label list becomes a 28-dimensional multi-hot vector, e.g. `[admiration, gratitude]` becomes 1s at those two positions and 0s elsewhere.
- **Model head and loss:** a linear layer with 28 outputs on top of BERT's `[CLS]` representation, a sigmoid per output, and binary cross-entropy (`BCEWithLogitsLoss`). Setting `problem_type='multi_label_classification'` makes Hugging Face use this loss.
- **Decision rule:** a label is predicted when its probability is at least a threshold. We report the default 0.5 and a single threshold tuned on the validation split (never on the test split).

Note for the project: these are emotions **expressed in the text**, which is a signal about wording, not proof of how a candidate feels. This matches the design rule in the repository README.

In [ ]:
def to_multi_hot(label_lists, num_labels=NUM_LABELS):
    """[[0, 4], [27]] -> float32 array of shape (n, num_labels) with 1s at the given ids."""
    vectors = np.zeros((len(label_lists), num_labels), dtype=np.float32)
    for row, labels in enumerate(label_lists):
        vectors[row, labels] = 1.0
    return vectors

example = raw['train'][0]
print([LABEL_NAMES[i] for i in example['labels']], '->', to_multi_hot([example['labels']])[0].astype(int))

## 3. Training, validation and test splits

We use the dataset's **official** train/validation/test splits, so results are comparable with the GoEmotions paper (its BERT baseline reaches a macro F1 of about 0.46 on the 28-label setup). Validation is used for model selection and threshold tuning; the test split is used once, at the end.

For the small run we take a seeded random subset of each split, so the run is reproducible.

In [ ]:
def take_subset(ds, n, seed):
    if n is None or n >= len(ds):
        return ds
    return ds.shuffle(seed=seed).select(range(n))

splits = {
    'train': take_subset(raw['train'], CONFIG['train_subset'], CONFIG['seed']),
    'validation': take_subset(raw['validation'], CONFIG['eval_subset'], CONFIG['seed']),
    'test': take_subset(raw['test'], CONFIG['eval_subset'], CONFIG['seed']),
}

# The splits must not share comments.
ids = {name: set(ds['id']) for name, ds in splits.items()}
assert not ids['train'] & ids['validation'], 'train/validation overlap'
assert not ids['train'] & ids['test'], 'train/test overlap'
assert not ids['validation'] & ids['test'], 'validation/test overlap'

split_summary = pd.DataFrame({
    name: {
        'comments': len(ds),
        'multi-label %': round(100 * np.mean([len(l) > 1 for l in ds['labels']]), 1),
        'labels with no positive example': int((to_multi_hot(ds['labels']).sum(0) == 0).sum()),
    }
    for name, ds in splits.items()
}).T
display(split_summary)

## 4. Pretrained BERT tokenizer and model for sequence classification

`bert-base-uncased` is the pretrained encoder (transfer learning). Only the new 28-output classification head starts from random weights, so the warning about newly initialised `classifier` weights is expected.

In [ ]:
from datasets import Sequence, Value

tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])

def preprocess(batch):
    encoded = tokenizer(batch['text'], truncation=True, max_length=CONFIG['max_length'])
    encoded['labels'] = to_multi_hot(batch['labels']).tolist()  # float targets for BCE loss
    return encoded

encoded = {
    name: ds.map(preprocess, batched=True, remove_columns=ds.column_names)
    # Store labels explicitly as float32, so no library turns the 0/1 values into integers.
    .cast_column('labels', Sequence(Value('float32')))
    for name, ds in splits.items()
}
print(encoded['train'])
print('labels feature:', encoded['train'].features['labels'])

token_lengths = np.array([len(x) for x in encoded['train']['input_ids']])
print(f"Tokens per comment: median {np.median(token_lengths):.0f}, "
      f"99th percentile {np.percentile(token_lengths, 99):.0f}, "
      f"truncated at {CONFIG['max_length']}: {(token_lengths >= CONFIG['max_length']).mean():.2%}")
print('Decoded example:', tokenizer.decode(encoded['train'][0]['input_ids']))

In [ ]:
id2label = dict(enumerate(LABEL_NAMES))
label2id = {name: i for i, name in id2label.items()}

model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG['model_name'],
    num_labels=NUM_LABELS,
    problem_type='multi_label_classification',
    id2label=id2label,
    label2id=label2id,
)
print(f'Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M')

## 5. Fine-tune, evaluate and save

### Multi-label metrics

Accuracy over single labels is misleading here (most labels are 0 for most comments), so we report:

| Metric | What it tells us |
|---|---|
| **micro F1** | F1 over all label decisions pooled; dominated by frequent labels |
| **macro F1** | mean of per-label F1; every emotion counts equally, so rare emotions matter |
| weighted F1, samples F1 | per-label F1 weighted by support; F1 per comment, averaged |
| micro / macro precision and recall | where errors come from (missed vs. extra labels) |
| subset accuracy | share of comments whose predicted label set is exactly right (strict) |
| Hamming loss | share of individual label decisions that are wrong |
| **macro ROC-AUC** (model selection) | threshold-free ranking quality, over labels present in the split; used to pick the best epoch because F1 at 0.5 can be 0 for every epoch early in training |

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def multilabel_metrics(probs, y_true, threshold):
    y_true = y_true.astype(int)
    y_pred = (probs >= threshold).astype(int)
    metrics = {
        'micro_f1': f1_score(y_true, y_pred, average='micro', zero_division=0),
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'weighted_f1': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'samples_f1': f1_score(y_true, y_pred, average='samples', zero_division=0),
        'micro_precision': precision_score(y_true, y_pred, average='micro', zero_division=0),
        'micro_recall': recall_score(y_true, y_pred, average='micro', zero_division=0),
        'macro_precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'macro_recall': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'subset_accuracy': accuracy_score(y_true, y_pred),
        'hamming_loss': hamming_loss(y_true, y_pred),
    }
    # ROC-AUC is only defined for labels with both positive and negative examples.
    positives = y_true.sum(axis=0)
    usable = (positives > 0) & (positives < len(y_true))
    metrics['macro_roc_auc'] = (
        roc_auc_score(y_true[:, usable], probs[:, usable], average='macro') if usable.any() else float('nan')
    )
    return {k: float(v) for k, v in metrics.items()}

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return multilabel_metrics(sigmoid(logits), labels, CONFIG['default_threshold'])

In [ ]:
class MultiLabelCollator(DataCollatorWithPadding):
    """Pads like DataCollatorWithPadding, then makes sure labels are float for BCEWithLogitsLoss."""

    def __call__(self, features):
        batch = super().__call__(features)
        batch['labels'] = batch['labels'].float()
        return batch

data_collator = MultiLabelCollator(tokenizer)

# Fail fast: check one real batch before spending time on training.
check_batch = data_collator([encoded['train'][i] for i in range(4)])
print({k: (tuple(v.shape), v.dtype) for k, v in check_batch.items()})
assert check_batch['labels'].dtype == torch.float32, 'labels must be float32 for multi-label loss'
assert check_batch['labels'].shape[1] == NUM_LABELS

steps_per_epoch = math.ceil(len(encoded['train']) / CONFIG['train_batch_size'])
warmup_steps = int(CONFIG['warmup_fraction'] * steps_per_epoch * CONFIG['num_epochs'])

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=CONFIG['learning_rate'],
    per_device_train_batch_size=CONFIG['train_batch_size'],
    per_device_eval_batch_size=CONFIG['eval_batch_size'],
    num_train_epochs=CONFIG['num_epochs'],
    weight_decay=CONFIG['weight_decay'],
    warmup_steps=warmup_steps,
    load_best_model_at_end=True,
    metric_for_best_model='macro_roc_auc',  # threshold-free, so it still ranks epochs when F1 at 0.5 is 0
    greater_is_better=True,
    save_total_limit=1,
    logging_steps=max(1, steps_per_epoch // 5),
    fp16=DEVICE == 'cuda',
    seed=CONFIG['seed'],
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded['train'],
    eval_dataset=encoded['validation'],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=CONFIG['early_stopping_patience'])],
)

start = time.time()
train_result = trainer.train()
train_minutes = (time.time() - start) / 60
print(f'Training took {train_minutes:.1f} min. Best checkpoint: {trainer.state.best_model_checkpoint}')

In [ ]:
# Validation metrics after each epoch.
history = pd.DataFrame([log for log in trainer.state.log_history if 'eval_micro_f1' in log])
cols = ['epoch', 'eval_loss', 'eval_macro_roc_auc', 'eval_micro_f1', 'eval_macro_f1', 'eval_samples_f1']
display(history[cols].round(4))

train_losses = pd.DataFrame([log for log in trainer.state.log_history if 'loss' in log])
ax = train_losses.plot(x='epoch', y='loss', label='train loss', figsize=(7, 3.5))
history.plot(x='epoch', y='eval_loss', label='validation loss', marker='o', ax=ax)
ax.set_title('Loss')
plt.tight_layout()
plt.show()

### Threshold tuning on the validation split

The sigmoid outputs for rare emotions are often below 0.5, so a single threshold chosen on validation usually improves F1. We pick the threshold with the best validation micro F1 and then apply it unchanged to the test split.

In [ ]:
val_out = trainer.predict(encoded['validation'])
val_probs, val_true = sigmoid(val_out.predictions), val_out.label_ids

threshold_grid = np.round(np.arange(0.05, 0.96, 0.05), 2)
sweep = pd.DataFrame([
    {'threshold': t, **multilabel_metrics(val_probs, val_true, t)} for t in threshold_grid
])
best_threshold = float(sweep.loc[sweep['micro_f1'].idxmax(), 'threshold'])
print(f'Best validation threshold: {best_threshold}')

ax = sweep.plot(x='threshold', y=['micro_f1', 'macro_f1'], marker='.', figsize=(7, 3.5), title='Validation F1 by threshold')
ax.axvline(best_threshold, color='grey', linestyle='--')
plt.tight_layout()
plt.show()

### Test evaluation (used once)

In [ ]:
test_out = trainer.predict(encoded['test'])
test_probs, test_true = sigmoid(test_out.predictions), test_out.label_ids.astype(int)

test_metrics = {
    'threshold_0.5': multilabel_metrics(test_probs, test_true, CONFIG['default_threshold']),
    'threshold_tuned': multilabel_metrics(test_probs, test_true, best_threshold),
}
display(pd.DataFrame(test_metrics).rename(columns={
    'threshold_0.5': 'threshold 0.5',
    'threshold_tuned': f'threshold {best_threshold} (tuned on validation)',
}).round(4))

In [ ]:
# Per-emotion results on the test split at the tuned threshold.
report = classification_report(
    test_true, (test_probs >= best_threshold).astype(int),
    target_names=LABEL_NAMES, zero_division=0, output_dict=True,
)
per_label = pd.DataFrame(report).T.loc[LABEL_NAMES]
per_label['support'] = per_label['support'].astype(int)
display(per_label.sort_values('f1-score', ascending=False).round(3))

### Save the model, tokenizer and results

The model folder is self-contained: the config holds the label names (`id2label`), and `inference_config.json` holds the decision threshold and `max_length` used here, so later code does not need this notebook.

In [ ]:
trainer.save_model(str(MODEL_DIR))  # best weights + config (with id2label)
tokenizer.save_pretrained(str(MODEL_DIR))

inference_config = {
    'task': 'multi_label_text_emotion_classification',
    'base_model': CONFIG['model_name'],
    'labels': LABEL_NAMES,
    'threshold': best_threshold,
    'max_length': CONFIG['max_length'],
}
(MODEL_DIR / 'inference_config.json').write_text(json.dumps(inference_config, indent=2))

results = {
    'run_name': RUN_NAME,
    'small_run': SMALL_RUN,
    'created_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    'config': CONFIG,
    'environment': ENV_INFO,
    'data': {name: len(ds) for name, ds in splits.items()},
    'training': {
        'minutes': round(train_minutes, 2),
        'epochs_run': float(trainer.state.epoch),
        'best_checkpoint': trainer.state.best_model_checkpoint,
        'best_validation_macro_roc_auc': trainer.state.best_metric,
    },
    'validation': {
        'tuned_threshold': best_threshold,
        'metrics_at_tuned_threshold': multilabel_metrics(val_probs, val_true, best_threshold),
    },
    'test': test_metrics,
    'test_per_label_f1': per_label['f1-score'].round(4).to_dict(),
    'model_dir': str(MODEL_DIR.relative_to(BASE_DIR)),
}
metrics_path = RESULTS_DIR / f'{RUN_NAME}_metrics.json'
metrics_path.write_text(json.dumps(results, indent=2))
per_label.to_csv(RESULTS_DIR / f'{RUN_NAME}_per_label.csv')

print('Saved model to   ', MODEL_DIR.resolve())
print('Saved metrics to ', metrics_path.resolve())
print(sorted(p.name for p in MODEL_DIR.iterdir()))

### Reload the saved model and predict

This checks that the saved folder works on its own and fixes the module's **input/output pattern**, which the speech and vision modules can follow:

```
predict_emotions(texts: list[str]) -> list[{
    'text': str,
    'labels': list[str],               # labels at or above the threshold
    'scores': dict[str, float],        # probability for every label
    'top': list[{'label', 'score'}],   # highest-scoring labels
}]
```

In [ ]:
def load_emotion_classifier(model_dir, device=DEVICE):
    model_dir = Path(model_dir)
    tok = AutoTokenizer.from_pretrained(model_dir)
    mdl = AutoModelForSequenceClassification.from_pretrained(model_dir).to(device).eval()
    cfg = json.loads((model_dir / 'inference_config.json').read_text())
    return tok, mdl, cfg

@torch.no_grad()
def predict_emotions(texts, tok, mdl, cfg, top_k=3):
    enc = tok(texts, truncation=True, max_length=cfg['max_length'], padding=True, return_tensors='pt')
    enc = enc.to(mdl.device)
    probs = torch.sigmoid(mdl(**enc).logits).float().cpu().numpy()
    outputs = []
    for text, p in zip(texts, probs):
        scores = {mdl.config.id2label[i]: round(float(s), 4) for i, s in enumerate(p)}
        ranked = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)
        outputs.append({
            'text': text,
            'labels': [label for label, s in ranked if s >= cfg['threshold']],
            'scores': scores,
            'top': [{'label': label, 'score': s} for label, s in ranked[:top_k]],
        })
    return outputs

clf_tok, clf_model, clf_cfg = load_emotion_classifier(MODEL_DIR)

# The reloaded model must reproduce the trainer's test probabilities.
sample_texts = splits['test']['text'][:32]
reloaded = np.array([[r['scores'][name] for name in LABEL_NAMES]
                     for r in predict_emotions(sample_texts, clf_tok, clf_model, clf_cfg)])
max_diff = np.abs(reloaded - test_probs[:32]).max()
print(f'Max probability difference vs. trainer predictions: {max_diff:.4f}')
assert max_diff < 1e-2, 'reloaded model does not match the trained model'

examples = [
    "I'm really excited about this role and grateful for the chance to interview.",
    "Honestly I'm not sure, that question confused me.",
    "We shipped the feature two weeks late, which was frustrating for the whole team.",
    "The API returns a JSON object with the user's id and name.",
]
for r in predict_emotions(examples, clf_tok, clf_model, clf_cfg):
    print(f"{r['text']}\n  labels: {r['labels']}\n  top: {r['top']}\n")

### Download the model and results

Files on a Colab runtime are deleted when it shuts down, so the next cell packs this run's model folder and results into `downloads/<run>.zip` and, on Colab, sends three files to your browser's downloads:

- `<run>_metrics.json` and `<run>_per_label.csv`: the evaluation results (small, sent first)
- `<run>.zip`: `model/` (weights, config, tokenizer, `inference_config.json`) plus `results/` (about 400 MB)

If the browser asks whether colab.research.google.com may download multiple files, allow it. You can also right-click the files in Colab's **Files** panel (folder icon on the left) and choose **Download**. To keep a copy in Google Drive as well, set `SAVE_TO_DRIVE = True` in the cell after it.

In [ ]:
import shutil
import zipfile

# Pack this run's model and results into one zip.
DOWNLOAD_DIR = BASE_DIR / 'downloads'
bundle_root = DOWNLOAD_DIR / RUN_NAME
if bundle_root.exists():
    shutil.rmtree(bundle_root)
shutil.copytree(MODEL_DIR, bundle_root / 'model')
(bundle_root / 'results').mkdir(parents=True)
run_result_files = [metrics_path, RESULTS_DIR / f'{RUN_NAME}_per_label.csv']
for f in run_result_files:
    shutil.copy2(f, bundle_root / 'results' / f.name)
zip_path = Path(shutil.make_archive(str(DOWNLOAD_DIR / RUN_NAME), 'zip', root_dir=bundle_root))
shutil.rmtree(bundle_root)

with zipfile.ZipFile(zip_path) as zf:
    print('\n'.join(sorted(zf.namelist())))
print(f'\nBundle: {zip_path.resolve()} ({zip_path.stat().st_size / 1e6:.0f} MB)')

if IN_COLAB:
    from google.colab import files
    for f in run_result_files:  # small files first, so results arrive even if the big download is slow
        files.download(str(f))
    files.download(str(zip_path))

In [ ]:
# Optional: also copy the zip and results to Google Drive (asks you to sign in to Drive).
SAVE_TO_DRIVE = False

if IN_COLAB and SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_dir = Path('/content/drive/MyDrive/interview-eval-agent/model-training-text')
    drive_dir.mkdir(parents=True, exist_ok=True)
    for f in [zip_path, *run_result_files]:
        shutil.copy2(f, drive_dir / f.name)
    print('Copied to', drive_dir)

## Next steps

1. **Small run passes:** every cell runs, the model folder and `results/bert-goemotions-small_metrics.json` exist, and the reload check passes. Scores are low on purpose here (2,000 training comments, 5 epochs): expect frequent labels such as *neutral*, *admiration* and *gratitude* to get non-zero F1, and macro ROC-AUC clearly above 0.5.
2. **Full run:** set `SMALL_RUN = False` and run all cells again, preferably on a GPU. A reasonable target is a test macro F1 close to the paper's BERT baseline (about 0.46) and a clearly higher micro F1.
3. **Later, not now:** wrap `load_emotion_classifier` / `predict_emotions` in an `ie_models` module with the same input/output shape and connect it to the backend's analysis service.